# YOLO Classification Inference on Colab
Notebook nay dung truc tiep model moi train trong Google Drive de chay phan loai anh.

In [ ]:
# Cell 1: Cai dat thu vien va mount Google Drive
!pip install ultralytics -q

import os
from google.colab import drive, files
from IPython.display import Image as IPyImage, display
from ultralytics import YOLO

drive.mount('/content/drive')
print('Drive da duoc mount.')

In [ ]:
# Cell 2: Nap model moi train tu Google Drive
DRIVE_ROOT = '/content/drive/MyDrive/YOLO_PBL5_Classification'
RUN_NAME = 'fruit_cls_v1'
IMGSZ = 224
PREFERRED_FORMAT = 'onnx'  # 'onnx' hoac 'pt'

WEIGHTS_DIR = os.path.join(DRIVE_ROOT, 'train_results', RUN_NAME, 'weights')
PT_MODEL_PATH = os.path.join(WEIGHTS_DIR, 'best.pt')
ONNX_MODEL_PATH = os.path.join(WEIGHTS_DIR, 'best.onnx')

if PREFERRED_FORMAT == 'onnx' and os.path.exists(ONNX_MODEL_PATH):
    MODEL_PATH = ONNX_MODEL_PATH
elif os.path.exists(PT_MODEL_PATH):
    MODEL_PATH = PT_MODEL_PATH
elif os.path.exists(ONNX_MODEL_PATH):
    MODEL_PATH = ONNX_MODEL_PATH
else:
    raise FileNotFoundError(
        f'Khong tim thay model trong {WEIGHTS_DIR}. Can best.pt hoac best.onnx.'
    )

model = YOLO(MODEL_PATH)
print(f'Dang dung model: {MODEL_PATH}')

In [ ]:
# Cell 3: Upload anh can phan loai
print('Chon mot hoac nhieu anh tu may tinh de phan loai:')
uploaded = files.upload()

if not uploaded:
    raise ValueError('Chua co anh nao duoc upload.')

uploaded_images = list(uploaded.keys())
print(f'Da tai len {len(uploaded_images)} anh.')

In [ ]:
# Cell 4: Chay inference va hien thi ket qua
RESULTS_ROOT = '/content/inference_results'

for img_path in uploaded_images:
    results = model.predict(
        source=img_path,
        imgsz=IMGSZ,
        save=True,
        project=RESULTS_ROOT,
        name='fruit_cls_v1',
        exist_ok=True,
        verbose=False
    )

    result = results[0]
    result_path = os.path.join(result.save_dir, os.path.basename(img_path))

    print(f'--- {img_path} ---')
    if hasattr(result, 'probs') and result.probs is not None:
        class_name = result.names[result.probs.top1]
        confidence = float(result.probs.top1conf)
        print(f'Du doan: {class_name} ({confidence:.2%})')
    else:
        print('Khong doc duoc xac suat du doan tu ket qua.')

    if os.path.exists(result_path):
        display(IPyImage(filename=result_path, width=500))
    else:
        print(f'Anh ket qua duoc luu trong: {result.save_dir}')